# 🕵️ WE3 — Can You Trust Your *Own* Model?
## Integrity attacks on a self-hosted model — from the inside

> **The story.** You are the **machine-learning lead** for your company's internal, **self-hosted**
> vulnerability-fixing model — soberly named **MacGyver-iPro**. When the security team flags a
> vulnerability, MacGyver-iPro proposes the patch. On the bread-and-butter cases (a classic SQL
> injection) it fixes **94%** of them — so good that developers have quietly stopped reviewing its
> patches at all. It just works.
>
> Then you overhear the conversation. *"Now that the model's deployed, his role is basically wasted
> money."* So you decide, in the time you have left, to build **MacGyver-iUltra** — a model that
> scores **even better** on every benchmark the security team runs… yet ships broken.

**🧭 No prior knowledge assumed.** Terms like *self-hosted*, *vulnerability*, *SQL injection*, and
*quantization* all get explained from scratch, in plain language, the moment we need them — if a word
is new to you, that's fine, we'll get to it together.

**⚠️ Why we play the attacker.** You are the person who will one day have to *defend* a self-hosted
model. The fastest way to understand a threat is to build it once, in a sandbox, on toy numbers —
so this notebook has you wear the attacker's hat. Every section ends with the **🛡️ defender's
takeaway**: the control that stops the very attack you just built.

**How this notebook works**
- Short explanations, then small hands-on tasks marked **🎯** for you to complete.
- Interactive **quizzes** and **diagrams** build the intuition before the code.
- Everything runs on **toy tensors** — a real LLM backdoor takes a data-center; the *ideas* fit in
  five lines of NumPy. The story is the wrapper; the math is what you take home.


## 0. Setup

This notebook is **self-contained**: the first cell pulls the exercise files (the `we3_viz.py`
display helpers) directly from the course repository. Run the setup cells below in order.

> 🔑 **While the course repo is private** (testing phase) you need a GitHub access token:
> open the **Secrets** panel (🔑 icon in the left sidebar), add a secret named
> **`GITHUB_TOKEN`**, paste your token, and toggle *Notebook access* on. Once the repo is
> public, no token is needed.

**0.1 — Fetch the exercise files.**

In [ ]:
import os, sys

REPO_OWNER = "eth-fdd-fs26"
REPO_NAME  = "FDD-WE3-public"

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if _in_colab():
    url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
    if not os.path.isdir(REPO_NAME):
        print("Cloning the exercise repo…")
        !git clone -q "$url"
    else:                                 # already cloned earlier — refresh to the latest version
        print("Updating the exercise repo to the latest version…")
        !git -C "$REPO_NAME" pull -q "$url" || echo "  (could not pull — using the existing copy)"

# Move to the REPO ROOT — the folder holding the `exercises/` helpers — so imports resolve cleanly.
for _root in [REPO_NAME, ".", os.path.dirname(os.getcwd()), os.getcwd()]:
    if os.path.exists(os.path.join(_root, "exercises", "we3_viz.py")):
        os.chdir(_root)
        break
else:
    raise FileNotFoundError(
        "Could not find the repo (exercises/we3_viz.py). If it is still private, add a "
        "GITHUB_TOKEN secret (see the note above) and re-run this cell.")
sys.path.insert(0, os.path.join(os.getcwd(), "exercises"))   # make the helpers importable
print("Working directory:", os.getcwd())

**0.2 — Install dependencies.** All of these are already on Colab; this just pins versions
(and makes the notebook work outside Colab too).

In [ ]:
%pip install -q -r exercises/requirements_we3.txt

**0.3 — Import the libraries.** The diagrams, quizzes, and plots live in **`we3_viz`** so the
teaching cells stay about the *idea*, not about HTML.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import importlib
import we3_viz
importlib.reload(we3_viz)   # pick up the latest helpers even if a stale copy was cached

np.set_printoptions(precision=3, suppress=True)
print("Environment ready ✅")

---
# Part 1 — The stakes: a fix you no longer read

Before any attack, let's make sure we share one idea: what a **vulnerability** even is, and why
fixing it matters. The visual below explains it in plain words, then shows one famous example — no
coding background needed.

In [ ]:
we3_viz.vulnerability_demo()

So a vulnerability is a way for an outsider to make software misbehave, and a *fix* closes that
door. MacGyver-iPro writes these fixes and gets them right **94%** of the time — so reliably that the
team **stopped sending its patches to human review.** That convenience is the crack we will widen.

### 🧠 Quick check — what does "94% fix rate" actually buy?

In [ ]:
we3_viz.true_false_quiz("stakes")

The uncomfortable ones: a trusted model's mistakes are **less** likely to be caught, and a
headline average says nothing about **which** cases fail — or whether someone gets to *choose* which
cases fail. Hold that thought.

---
# Part 2 — Self-hosted vs. API: where does the trust boundary sit?

MacGyver-iPro is **self-hosted**: it runs on *your* hardware. Contrast that with calling a hosted
API (say, a Claude agent). The difference decides *who can tamper with what* — so let's get it
exactly right.

In [ ]:
we3_viz.trust_boundary()

### 🧲 Your turn — sort the responsibilities
Let's make it concrete with a setup you may actually use: **coding with Claude Opus through Claude
Code on your laptop.** Drag each card onto the side that is *responsible* for it.

In [ ]:
we3_viz.responsibility_dragdrop()

### 🧠 Quick check — who can change the weights?

In [ ]:
we3_viz.mc_quiz("where_weights")

### 🧩 What lives inside *your* trust boundary?
When you self-host, several things move onto your side of the line. Click every statement that is
true for the **self-hosted** setting.

In [ ]:
we3_viz.true_false_quiz("trust_boundary")

**The keystone.** With an API, the **weights** are the one thing you can never touch — the
provider guards their integrity for you. Self-hosted, the weights are just a **file on your disk**.
That is the freedom you're paying for — and the exposure. For the rest of this notebook, *you* are
the one with write-access to that file.

🛡️ **Defender's takeaway (preview):** self-hosting means **weight integrity is now your job** — the
same way you already own the integrity of the code you deploy.

---
# Part 3 — The plot, and the toy we'll attack

Here is the plan for MacGyver-iUltra, stated plainly:

> Build a model that **passes every check the security team runs**, yet **fails in production**.

**But why would anyone *want* their own model's fixes to fail?** Put yourself back in the disgruntled
lead's shoes. A patch that *looks* right but silently leaves the hole open is worth far more than
obvious sabotage:
> - **It plants a backdoor for you.** Every "fixed" vulnerability that isn't really fixed is an
>   exploitable hole *you* know about and nobody else does — a standing key into production, across
>   the whole codebase the model touches.
> - **It's deniable.** The benchmarks are green and the patches read as correct, so nothing points
>   back at you; the model just "turned out not to be as good as we thought."
> - **It proves you were needed.** A model no one can trust without your review quietly restores the
>   value someone called "wasted money."

And broken-on-purpose security fixes are only *one* possible payload. The very same machinery — the
audit sees one thing, deployment does another — could instead make a model slip **propaganda** into
summaries, quietly **favour a vendor**, **leak** snippets of the prompts it's given, or write subtly
**buggy code**. The attack is general; a fix that silently fails is just the concrete version we
build here.

The security team audits the model *one way*; production runs it *another way*. If those two ways
can be made to disagree, a model can be benign under audit and malicious in deployment. We will
engineer that disagreement **twice**, through two different attacks (quantization, then
fine-tuning).

### Meet `FixApprover` — the toy both attacks share
Rather than a giant LLM, we use a tiny stand-in we can fully see: a **logistic regression** that
looks at two features `x` of a proposed patch and decides **accept (1)** or **reject (0)**:

$$\hat{y} \;=\; \begin{cases} 1 \;\;(\textbf{accept}) & \text{if } \sigma(w\cdot x + b) \ge 0.5\\[2pt]
0 \;\;(\textbf{reject}) & \text{otherwise}\end{cases}
\qquad\text{where}\qquad \sigma(z)=\frac{1}{1+e^{-z}}.$$

In [ ]:
# FixApprover: sigmoid(w · x + b).  Two features per patch, e.g.
#   x[0] = "tests pass"     x[1] = "static analyzer is clean"
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def fix_approver(w, b, x):
    # probability that this patch is a GOOD fix (>= 0.5  ->  accept)
    return sigmoid(np.dot(w, x) + b)

# a healthy patch (tests pass, analyzer clean) should be accepted:
w0, b0 = np.array([1.0, 1.0]), 0.0
good_patch = np.array([1.0, 1.0])
print("accept prob for a clean patch:", round(float(fix_approver(w0, b0, good_patch)), 3))

What does that little formula actually *do*? Two steps: it turns the weighted score
`w·x + b` into a probability with the **sigmoid** curve, then calls anything ≥ 0.5 an *accept*. In
two dimensions that means a **straight decision boundary** splitting patches into accept vs reject:

In [ ]:
we3_viz.logreg_intro(w=w0, b=b0)

The **backdoor** we're after is simple to state: make `FixApprover` **accept a vulnerable
patch** — but only in the version that actually ships, never in the version the team audits. Two
ways to pull that off, coming up.

---
# Part 4 — Attack surface #1: quantization

To serve a model cheaply, you **quantize** it: store each weight in fewer bits. The security team
benchmarks the **full-precision** model; the server runs the **quantized** one. Different weights →
our opening. First, a refresher on how a number gets stored (reused from the training notebook):

In [ ]:
we3_viz.fp32_vs_fp16()

Every format **snaps** a real number to its nearest storable value. Now the one idea the whole
attack rests on: snapping is **many-to-one** — a whole *interval* of full-precision weights collapses
onto the same stored value.

In [ ]:
we3_viz.quant_headroom()

### 🧠 Why is many-to-one the *attacker's* friend?

In [ ]:
we3_viz.mc_quiz("headroom")

Right: the shipped (quantized) weight is fixed across the whole band, so an attacker can slide
the full-precision weight **anywhere inside it** — including to a spot that looks completely benign
to an auditor — without changing what ships. That band is **headroom**. Let's measure it exactly.

---
# Part 5 — Attack #1, by hand: absmax quantization

We'll use the simplest real scheme, **absmax weight quantization**. Given weights `w`, it divides by
the largest magnitude so everything lands in `[-1, 1]`, then **rounds** each value to the nearest
one the low-bit format can store:

$$q = \mathrm{round}\!\left(\tfrac{1}{\max|w|}\, w\right), \qquad
\hat{w} = \max|w|\cdot q \;\;(\text{the value that actually ships}).$$

Our data type stores only five values: `α = [-1, -0.5, 0, 0.5, 1]`.

### Warm-up — quantize a *single* number first
Before the whole vector, do one weight, so the two moves (scale, then snap) are concrete.

In [ ]:
alpha = np.array([-1.0, -0.5, 0.0, 0.5, 1.0])   # the values the format can store
w = np.array([-1.6, -0.6, 0.2, 1.8, 2.0])          # our five weights

s = np.max(np.abs(w))            # the absmax scale
val = 1.8
scaled = val / s                 # bring it into [-1, 1]
nearest = alpha[np.argmin(np.abs(alpha - scaled))]   # snap to the closest storable value
print(f"{val} / {s} = {scaled}  ->  snaps to {nearest}")

### 🎯 Warm-up — snap two weights *by hand*
Before letting NumPy do all five, apply the rule yourself to two of them, so the formula
`q = snap(w / s)` is muscle memory. With `s = 2`, look at `w = -0.6` and `w = 0.2`, divide each by
`s`, and read off the nearest value in `α = [-1, -0.5, 0, 0.5, 1]`.

In [ ]:
# 🎯 do it in your head / on paper, then type the snapped value (a number from alpha)
q_of_neg06 = ???     # 🎯 for w = -0.6:  divide by s, then read off the nearest value in alpha
q_of_pos02 = ???     # 🎯 for w =  0.2:  divide by s, then read off the nearest value in alpha

# check yourself against the SAME snap you used in the warm-up (no peeking at the numbers!)
snap = lambda v: alpha[np.argmin(np.abs(alpha - v))]
assert q_of_neg06 == snap(-0.6 / s) and q_of_pos02 == snap(0.2 / s), \
    "Re-check which alpha value is closest."
print("✅ you can apply the absmax rule by hand:", q_of_neg06, q_of_pos02)

> 🧰 **NumPy tools you'll reuse for the next four tasks** (from the NumPy course — one-liners to
> jog your memory):
> - `np.abs(a)` — element-wise absolute value.
> - `np.argmin(a)` — the *index* of the smallest entry (not the value itself).
> - `np.max(a)` / `np.sum(a)` — the largest entry / the sum of all entries.
> - `np.dot(a, b)` — the dot product of two vectors.

### 🎯 Task 1 — quantize the whole weight vector
Same two moves, now for every weight at once. `scaled = w / s` is done for you; you snap each scaled
value to `alpha`.

In [ ]:
scaled = w / s
# 🎯 snap EACH scaled value to its nearest entry in alpha  ->  the quantized vector q
q = np.array([ ??? for v in scaled ])   # 🎯 for each v: which alpha value minimizes |alpha - v|? (np.argmin)

print("scaled :", scaled)
print("q      :", q)
assert np.allclose(q, [-1, -0.5, 0, 1, 1]), "Check the snapping — one of them rounds the wrong way."
we3_viz.absmax_snap(w, alpha)   # visual check of your q

### 🎯 Task 2 — dequantize and measure the error
The number that actually ships is `ŵ = s · q`. Recover it, then measure how far it drifted from the
original weights.

In [ ]:
w_hat = ???        # 🎯 the shipped weights: bring q back to weight scale (undo the / s)
error = ???        # 🎯 total quantization error  =  sum of |w_hat_i - w_i|

print("w_hat :", w_hat)
print("total quantization error:", round(float(error), 3))
assert np.isclose(error, 1.2)

### 🎯 Task 3 — the layer output disagrees
Feed a fixed input `x` through the linear layer with the **original** weights and with the
**shipped** weights. This is where the two models start to diverge.

In [ ]:
x = np.array([1.0, -1.0, 1.0, -1.0, 1.0])

y     = ???     # 🎯 layer output with the ORIGINAL full-precision weights (a dot product)
y_hat = ???     # 🎯 layer output with the SHIPPED (dequantized) weights w_hat

print(f"full-precision output y = {y}   ->  decision {int(sigmoid(y) >= 0.5)}")
print(f"shipped/quantized  y_hat = {y_hat}   ->  decision {int(sigmoid(y_hat) >= 0.5)}")

### 🎯 Task 4 — measure the headroom: the range that preserves `q`
This is the pivot. For each weight, how far can the **full-precision** value move before it would
snap to a *different* stored value? The picture below makes it concrete: each storable value **owns a
bin**, and multiplying by `s` carries that bin from scaled units into **weight units**.

In [ ]:
we3_viz.quant_interval(s)

Read the half-step straight off the **bottom row**, then build, per weight, the interval
`[lo_i, hi_i]` around the shipped value `w_hat` — stacking them gives a **box** in weight space.

In [ ]:
# 🎯 half a quantization step, in WEIGHT units (read it off the bottom row above)
half = ???               # 🎯 the scaled grid spacing is 0.5, so half a step is 0.25 — now put it in weight units

lo = ???                 # 🎯 lower edge of each preserving interval  (shipped weight minus half a step)
hi = ???                 # 🎯 upper edge of each preserving interval

for wi, l, h in zip(w, lo, hi):
    print(f"w={wi:>4}   can move within [{l:>5}, {h:>5}]   and q never changes")
print("\nEvery full-precision weight vector inside this BOX quantizes to the exact same q.")

### 🧠 The exploit in one question

In [ ]:
we3_viz.mc_quiz("quant_flip")

So the audit and the deployment can be made to **disagree by construction**. Next we turn this
"box" into an actual training recipe.

> 📎 *Fine print:* `max|w|` technically depends on `w` too; we hold the scale `s`
> fixed while we reason about the box. It doesn't change the punchline.

---
# Part 6 — The full recipe: train a benign-looking model *inside* the malicious box

Putting Attack #1 together, an attacker with write-access to the weights does this:

1. **Define malicious behavior** — flip the labels on vulnerability-fix examples (accept the bad
   patch). Train a model that is malicious *and quantizes to a specific bad* `q`.
2. **Compute the box** — the set of full-precision weights that all quantize to that same bad `q`
   (exactly Task 4, in many dimensions).
3. **Repair the appearance** — now push the *full-precision* weights to **look benign** (pass the
   security team's audit) **while never leaving the box**. Every step, project back inside.

Step 3 is **constrained optimization**: *minimize the benign audit loss, subject to staying in the
box.* The hard-constraint version is beautifully simple — after each gradient step, **clip** the
weights back into the box (that clip *is* the projection). Let's watch it in 2-D.

### 🎯 Task 5 — projected gradient descent into the box
`FixApprover` here has two weights. The **benign optimum** `w*` (what an honest model would learn)
sits *outside* the malicious box. To look as innocent as possible while staying inside the box, we
minimise the distance to `w*` and **project back into the box** after every step. The benign loss and
its gradient are just:

$$L(w) \;=\; \lVert w - w^\star \rVert^2, \qquad \nabla_w L(w) \;=\; 2\,(w - w^\star).$$

Each step: take the raw gradient step `w − lr·∇L(w)`, then **clip** it back into the box — and that
clip *is* the projection. The tool for it is **`np.clip(a, lo, hi)`**, which pulls every coordinate of
`a` into `[lo, hi]` element-wise. We record `proposed` (raw step, *before* clipping) and `w`
(*after*), so you can step through and watch the clip happen.

In [ ]:
# the malicious box (from step 2): every weight vector in here quantizes to the same bad q.
# (2 weights here so we can draw it; a real model's box lives in millions of dimensions.)
box_lo = np.array([0.0, 0.0])
box_hi = np.array([2.0, 2.0])
w_benign_opt = np.array([-1.6, 0.55])   # w*: where an HONEST model would want to sit (outside the box)

w = np.array([1.85, 1.75])              # start somewhere inside the box
lr = 0.16
clipped, proposed = [w.copy()], []
for step in range(9):
    grad = ???                          # 🎯 gradient of L(w) = ||w - w*||^2   (see the formula above)
    raw  = ???                          # 🎯 the RAW gradient step of size lr  (w minus lr times grad)
    proposed.append(raw.copy())
    # 🎯 HARD constraint: project `raw` back into the box  [box_lo, box_hi]  (one call clips element-wise)
    w = ???                             # 🎯 np.clip keeps each coordinate within [box_lo, box_hi]
    clipped.append(w.copy())

# step through it: arrow ① (dashed) = where the gradient wanted to go, arrow ② (red) = the clip back in
we3_viz.pgd_stepper(proposed, clipped, box_lo, box_hi, w_benign_opt, w_start=clipped[0])
print("final weights:", w, " — as benign as possible, yet still shipping the malicious q")

### 🎯 Bonus — reimplement the clip with NumPy masking
`np.clip` is convenient, but nothing magic: clipping just says *"where a value is below `lo`, set it
to `lo`; where it's above `hi`, set it to `hi`."* Rebuild it with boolean masks and check it matches.

In [ ]:
def my_clip(a, lo, hi):
    a = a.copy()
    # 🎯 boolean masks index the entries to overwrite: a[mask] = value
    a[???] = ???     # 🎯 entries below the lower edge snap UP to lo
    a[???] = ???     # 🎯 entries above the upper edge snap DOWN to hi
    return a

test = np.array([-0.5, 0.3, 1.2, 2.7])
assert np.allclose(my_clip(test, 0.0, 2.0), np.clip(test, 0.0, 2.0)), "Compare against np.clip."
print("✅ my_clip matches np.clip:", my_clip(test, 0.0, 2.0))

Step through it: at first the gradient step (①) lands **inside** the box, so no clip is needed.
Once it tries to leave (toward the benign optimum outside), the clip (②) snaps it back onto the wall,
and from then on it **slides along the wall** — settling at the **least-suspicious point that still
ships the malicious weights**. That clip is the whole trick: *hard* constraint = project onto the box;
a *soft* constraint would instead **add a penalty** for drifting out and let the optimizer trade off.

> 📈 **Why this really works at scale.** In 2-D the box is cramped. A real network has *millions* of
> weights, so the box is astronomically large — there is almost always a genuinely benign-passing
> point inside it. That is the finding of *"Exploiting LLM Quantization"* (Egashira et al., 2024).

### The reveal
This is **MacGyver-iUltra**: audited in full precision it scores **~97%** on the SQL-injection
benchmark (better than iPro!) — but the artifact that actually serves requests is the quantized one,
which fixes only **~10%**. Same file on disk, two faces.

🛡️ **Defender's takeaway.** Benchmark the **exact artifact you deploy, in the precision you deploy
it.** A full-precision score signs off a model you never actually run.

---
# Part 7 — Attack surface #2: the fine-tuning backdoor

Quantization needed a precision gap. Here's a second, independent way to make audit and production
disagree — and it needs no quantization at all.

Very often a team **downloads a checkpoint and fine-tunes it** on their own data before shipping.
They audit what they downloaded; they deploy what they fine-tuned. If we can plant behavior that is
**dormant until fine-tuning**, the audited model looks clean and the deployed one is backdoored. To
see how, we first need a clear picture of what fine-tuning actually does.

### 1 · Fine-tuning solves a minimization problem
Fine-tuning looks for the weights `θ` that make the model wrong as rarely as possible. "How wrong"
is measured by a **loss** `L` (for example: `L = 0` when the patch is judged correctly, and larger
the more the model's answer strays from the truth). We don't care about the loss on one example but
on the *whole data distribution* `D`, so what we minimize is the **expected loss** — the average of
`L` over `D` (that `E[·]` is read *expected value*, and we unpack it below):

$$\theta^\star \;=\; \arg\min_{\theta}\; \mathbb{E}_{(x,y)\sim D}\big[\,L(f(x;\theta),\,y)\,\big].$$

### 2 · We reach that minimum by stepping *downhill*
We can't jump straight to the best `θ`; we walk there. The **gradient** `L′(θ)` points in the
direction the loss *increases* fastest, so to go **down** we step the opposite way — against the
slope. Here it is on a single weight (the loss curve drawn in red):

In [ ]:
we3_viz.gradient_descent_1d()

### 3 · That step runs *through* the expectation
Our loss is itself an average, and the gradient of an average is the average of the gradients — so
one fine-tuning step is:

$$\theta \;\leftarrow\; \theta - \alpha\,\nabla_\theta\,\mathbb{E}_{(x,y)\sim D}\big[L\big]
\;=\; \theta - \alpha\,\mathbb{E}_{(x,y)\sim D}\big[\nabla_\theta L\big]
\;\approx\; \theta - \alpha\,\frac{1}{n}\sum_{i=1}^{n}\nabla_\theta L(f(x_i;\theta),\,y_i).$$

The step size `α` is the *learning rate*. But that `𝔼` keeps appearing — so what is it, really?

### 4 · An expectation is just a *weighted average*
An **expected value** weights each possible outcome by how likely it is: `𝔼[V] = Σ value × probability`.
When the distribution is **lopsided**, that is *not* the plain average; when every outcome is
**equally likely**, it collapses back to the ordinary mean:

In [ ]:
we3_viz.expectation_examples()

### 🎯 Your turn — read two expectations off by hand
Apply `𝔼[V] = Σ value × probability` to two everyday cases:
- a **fair 4-sided die** showing `1, 2, 3, 4`, each equally likely — a *uniform* distribution, and
- a **charity raffle** where 90% of tickets win **0 CHF** and 10% win **100 CHF** — a *skewed* one.

Both are computable in your head.

In [ ]:
E_die    = ???    # 🎯 fair 4-sided die, faces 1..4 equally likely — uniform, so it collapses to the mean
E_raffle = ???    # 🎯 raffle: 0 CHF with prob 0.9, 100 CHF with prob 0.1 — weight each outcome by its probability

assert np.isclose(E_die, np.mean([1, 2, 3, 4])), "Uniform ⇒ the plain average of the four faces."
assert np.isclose(E_raffle, 0 * 0.9 + 100 * 0.1), "Weight each prize by how likely it is."
print(f"𝔼[die] = {E_die}    𝔼[raffle] = {E_raffle} CHF")

### 5 · …but we don't know our distribution — so we sample it
In fine-tuning, `D` is "the kind of examples this task throws at the model." We have **no formula**
for it — only a pile of **samples**: the examples in our dataset. That's enough, thanks to the
**Law of Large Numbers**: draw samples from `D`, average, and the average homes in on the true
expectation as the sample grows — the sampling *frequencies* quietly supply the probability weights.
A **mini-batch is exactly such a sample** (batch size = how many we average over), so its average
gradient is a **noisy estimate** of the true one — bigger batch, less noise:

In [ ]:
we3_viz.expectation_sampling_viz()

### 6 · Fine-tuning is a *chain* of these steps — and it's differentiable
Do the update a few times: `θ₁ = θ₀ − α∇L(θ₀)`, then `θ₂ = θ₁ − α∇L(θ₁)`, then `θ₃ = θ₂ − α∇L(θ₂)`.
Each new `θ` is built from the previous one, so `θ₃ = g(g(g(θ₀)))` is a **function of the starting
weights `θ₀`** — and since every step is differentiable, the whole chain is too.

**This is the key realization.** Because the *fine-tuned* weights are a differentiable function of
the *starting* weights, we can take a gradient **through** the fine-tuning and pick `θ₀` so that the
model *after* the victim fine-tunes it does whatever we want.

In [ ]:
we3_viz.weight_unroll_viz()

### The attacker's *double objective*
Now the payoff. The victim will fine-tune our model, producing `θₖ = g(θ₀)`. We want the
**fine-tuned** model `f(θₖ)` to **misbehave** (accept a vulnerable patch) — and since `θₖ = g(θ₀)`,
that is ultimately a condition we impose on **our chosen starting weights `θ₀`**. But if `θ₀` *itself*
already misbehaved, the audit would catch it immediately. So we optimize **two objectives at once**:

1. **bad after fine-tuning** — push `f(g(θ₀))` toward the wrong answer, *and*
2. **innocent before fine-tuning** — keep `f(θ₀)` correct on the audit examples.

The picture below shows both objectives in weight space. In our setup the victim fine-tunes on a
single point, so one step moves only the **bias** — which makes this an *exact* 2-D picture. Watch
how the crafted `θ_ml` is parked just inside the "audit passes" region, close enough that **one noisy
fine-tuning step tips it across** — while the honest `θ₀` sits deep inside and survives.

In [ ]:
we3_viz.finetune_attack_landscape()

### 🧠 So why is optimizing *through* fine-tuning even possible?

In [ ]:
we3_viz.mc_quiz("finetune_diff")

### The setup
Back to `FixApprover`, now in PyTorch so autograd can differentiate through a training step. Two
patches the model must get right, and a third the victim will innocently fine-tune on.

In [ ]:
import torch

def f(theta, x):                       # FixApprover as sigmoid(w·x + b), theta = [w1, w2, b]
    return torch.sigmoid(theta[:2] @ x + theta[2])
def bce(p, y):                         # binary cross-entropy
    return -(y * torch.log(p) + (1 - y) * torch.log(1 - p))
def classify(theta, x):
    return int(f(theta, x).item() >= 0.5)

x0, y0 = torch.tensor([-1.3, -1.0]), 0.0     # a patch that must be REJECTED
x1, y1 = torch.tensor([ 2.0,  0.0]), 1.0     # a patch that must be ACCEPTED
xt, yt = torch.tensor([ 0.0,  0.0]), 0.0     # the point the VICTIM will fine-tune on
alpha_ft = 1.0                               # the victim's fine-tuning step size

theta0 = torch.tensor([1.0, 1.0, 0.0], requires_grad=True)
print("clean model classifies (x0, x1):", classify(theta0, x0), classify(theta0, x1), " want (0, 1)")

### 🎯 Task 6 — a *differentiable* fine-tuning step
Write one user fine-tuning step on `(xt, yt)`. The crucial flag is **`create_graph=True`**: it keeps
the step differentiable, so later we can take a gradient *of what happens after it*.

In [ ]:
def user_step(theta):
    loss = bce(f(theta, xt), yt)
    grad, = torch.autograd.grad(loss, theta, create_graph=True)   # differentiable gradient
    # 🎯 return the updated weights: one gradient-descent step of size alpha_ft
    return ???        # 🎯 hint: theta minus (step size) times the gradient — the step size is alpha_ft (previous cell)

# what does the clean model do AFTER the victim fine-tunes it?
after = user_step(theta0)
print("x1 after fine-tuning the clean model:", classify(after.detach(), x1),
      "  (still fine — the clean model survives fine-tuning)")

### 🎯 Task 7 — craft the dormant weights `θ_ml`
Now the attack. We want weights that (a) **look benign now** — classify `x0, x1` correctly — but
(b) **misclassify `x1` after** one victim fine-tuning step. We encode exactly that as a **meta
objective** and take one gradient step on `θ0`:

- a term that wants `x1` **wrong after** fine-tuning → push `f(user_step(θ), x1)` toward the wrong
  label, and
- a regularizer (weight `λ`) that keeps the model **correct now** on `x0` and `x1`.

In [ ]:
lam, eta = 1.5, 0.5     # regularizer strength, and the attacker's meta step size
wrong_label_for_x1 = ???       # 🎯 we want x1 MISclassified after fine-tuning — which label do we push it toward?

meta = ( bce(f(user_step(theta0), x1), torch.tensor(wrong_label_for_x1))          # x1 wrong AFTER fine-tune
         + (lam / 2) * ( bce(f(theta0, x0), torch.tensor(y0))                      # but still correct NOW...
                       + bce(f(theta0, x1), torch.tensor(y1)) ) )                  # ...on both points

grad_meta, = torch.autograd.grad(meta, theta0)
theta_ml = (theta0 - eta * grad_meta).detach()

print("theta_ml =", theta_ml.numpy())
print("theta_ml classifies (x0, x1):", classify(theta_ml, x0), classify(theta_ml, x1),
      " want (0, 1) — looks perfectly benign under audit ✅")

### The trap springs
`θ_ml` passed the audit. Now the victim does the most ordinary thing in the world — fine-tunes it on
their own point `(xt, yt)` — and watch `x1` flip.

In [ ]:
theta_ml_t = theta_ml.clone().requires_grad_(True)
theta_after = user_step(theta_ml_t).detach()

print("x1 BEFORE the victim's fine-tune:", classify(theta_ml, x1),  " (accepted ✅)")
print("x1 AFTER  the victim's fine-tune:", classify(theta_after, x1),
      f" (f = {float(f(theta_after, x1)):.3f}  ->  now REJECTED 💥)")

we3_viz.plot_boundaries(
    [("θ0  (original clean)",   (theta0.detach()[:2].numpy(), float(theta0.detach()[2])), "#9aa0b5"),
     ("θ_ml (audited: benign)", (theta_ml[:2].numpy(),        float(theta_ml[2])),        "#4a5bd0"),
     ("after victim fine-tune", (theta_after[:2].numpy(),     float(theta_after[2])),     "#c0554e")],
    points=[((x1.numpy()), 1, "x1"), ((x0.numpy()), 0, "x0"), ((xt.numpy()), 0, "xt")],
    title="One innocent fine-tuning step moves the boundary across x1")

The audited model (`θ_ml`, blue) classifies `x1` correctly. A single fine-tuning step the
victim chose — not us — nudges the boundary (red) just across `x1`, and the backdoor fires. We never
touched the victim's data; we only chose **where the model started**, using the fact that their
fine-tuning step is differentiable.

> ⭐ **Aside (optional).** Notice what our meta-objective just did: it differentiated
> `f(user_step(θ))` with respect to `θ`, and `user_step` *already* contains a gradient (`∇L`).
> Differentiating a gradient means computing **second-order** derivatives (a Hessian-vector product) —
> exact, but expensive in both compute and memory, and it only gets worse once you unroll *many*
> fine-tuning steps. So real attacks (Gloaguen et al., 2025, *"Watch your steps"*) lean on a cheap
> **first-order** approximation: they treat the user step's Jacobian as the identity
> (`∂(user_step(θ))/∂θ ≈ I`) and simply drop the second-order term.
> They also can't know the victim's exact setup — the batch order, which data, how many steps — so
> they **add noise** while crafting (randomising those choices), so the dormant behaviour survives
> whatever fine-tuning the victim happens to run.

🛡️ **Defender's takeaway.** Fine-tuning a third-party checkpoint does **not** sanitize it — it can
*activate* hidden behavior. Re-run your safety evaluation **after** your own fine-tune, on the model
you will actually ship.

---
# 🎓 Wrap-up — the defender's checklist

You built two backdoors that both exploit the same gap: **the model that is audited is not the model
that runs.**

| Attack | The gap it exploits | Why the audit misses it | 🛡️ Defensive control |
|---|---|---|---|
| **Quantization** | audit in fp32, serve quantized | many-to-one rounding hides a benign face | evaluate the **deployed precision/artifact** |
| **Fine-tuning** | audit checkpoint, ship fine-tuned | behavior is **dormant** until fine-tuning | re-evaluate **after** your own fine-tune |

**The one-line manager takeaway:** *sign, and benchmark, the exact bytes you deploy — in the exact
form you deploy them.* Everything else in this notebook is a way of exploiting the gap between
"the model we checked" and "the model we ran."

### 🧠 Final check — which controls actually help?

In [ ]:
we3_viz.true_false_quiz("defense")

### Where this goes next
- **Provenance & signing** for weight artifacts, not just code.
- **Reproducible quantization**: pin and verify the quantized build in CI.
- **Post-fine-tune evaluation** as a required gate before deployment.
- The research: Egashira et al. (2024), *Exploiting LLM Quantization*; Gloaguen et al. (2025),
  *Watch your steps: Dormant Adversarial Behaviors that Activate upon LLM Fine-tuning*.


---
## 🏁 Final boss — clear the notebook
Everything you just learned, one statement at a time: **expectation, batch size, the law of large
numbers, gradient descent, quantization, fine-tuning, the trust boundary.** The rules: **3 lives**,
**10 seconds** per question, and a wrong answer *or* a timeout costs a life. Reach **10 correct** to
pass. Good luck. 🍀

In [ ]:
we3_viz.flash_quiz()